In [1]:
import pandas as pd

def detect_swing_highs_lows(df, n=5):
    """
    Détecte les swing highs et swing lows dans un DataFrame pandas.
    n : nombre de bougies à gauche et à droite pour valider le pivot
    """
    df = df.copy()
    df['swing_high'] = False
    df['swing_low'] = False
    
    for q in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[q - n:q + n + 1]
        if df['High'].iloc[q] == high_range.max():
            df.loc[df.index[q], 'swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[q - n:q + n + 1]
        if df['Low'].iloc[q] == low_range.min():
            df.loc[df.index[q], 'swing_low'] = True
            
    return df

# Exemple d'utilisation avec des données fictives
# data = {
#     'high': [10, 12, 15, 13, 11, 14, 18, 16, 12, 10],
#     'low':  [8,  9,  10, 8,  7,  9,  12, 10, 8,  6]
# }
# df_prices = pd.DataFrame(data)
# result_df = detect_sent = detect_swing_highs_lows(df_prices, n=2)
# print(result_df)

df = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df = df.loc['2026-09-14' : '2026-9-16']

timezone = "Europe/London"
df.index = pd.to_datetime(df.index, utc=True).tz_convert(timezone)

df_result = detect_swing_highs_lows(df, n=5)
for index, row in df_result.iterrows():
    print(row)

Open          207.503
High          207.822
Low           207.455
Close         207.706
Tickvol         29155
swing_high      False
swing_low       False
Name: 2026-09-14 01:00:00+01:00, dtype: object
Open          207.704
High          207.937
Low           207.658
Close         207.795
Tickvol         20393
swing_high      False
swing_low       False
Name: 2026-09-14 02:00:00+01:00, dtype: object
Open          207.796
High          208.182
Low           207.796
Close         208.128
Tickvol         10864
swing_high      False
swing_low       False
Name: 2026-09-14 03:00:00+01:00, dtype: object
Open          208.128
High          208.191
Low           208.018
Close         208.053
Tickvol         11685
swing_high      False
swing_low       False
Name: 2026-09-14 04:00:00+01:00, dtype: object
Open          208.052
High           208.07
Low           207.893
Close         207.978
Tickvol         16119
swing_high      False
swing_low       False
Name: 2026-09-14 05:00:00+01:00, dtype: ob

In [3]:
import numpy as np
import pandas as pd


def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    # 1. Détection des Swings avec décalage de 'n' bougies pour éviter le biais de futur
    # df["is_swing_high"] = (
    #     (df["High"] == df["High"].rolling(2 * n + 1, center=True).max())
    # ).shift(n)
    # df["is_swing_low"] = (
    #     (df["Low"] == df["Low"].rolling(2 * n + 1, center=True).min())
    # ).shift(n)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = "NEUTRAL"

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


# --- SIMULATION ET VÉRIFICATION DU CYCLE ---
np.random.seed(42)
# Génération d'un krach suivi d'un retournement brutal (pour tester l'interdiction de saut d'état)
price_trend = (
    [100] * 10
    + [100 + x * 1.5 for x in range(10)]  # Hausse -> Devrait passer Bullish
    + [115 - x * 4.0 for x in range(10)]  # Chute brutale -> Doit d'abord passer par Neutral
    + [75 + x * 2.0 for x in range(10)]  # Reprise
)

data = {
    "High": [p + np.random.uniform(0.5, 1.2) for p in price_trend],
    "Low": [p - np.random.uniform(0.5, 1.2) for p in price_trend],
    "Close": [p + np.random.uniform(-0.3, 0.3) for p in price_trend],
}
df_test = pd.DataFrame(data)

df_result = implement_market_structure_states_strict(df_test, n=2)
print(df_result)

# Affichage des transitions d'états
df_result["state_changed"] = df_result["market_state"] != df_result[
    "market_state"
].shift(1)
# print(df_result[df_result["state_changed"]][["close", "market_state"]])


          High         Low       Close  is_swing_high  is_swing_low  \
0   100.762178   99.414573  100.217862          False         False   
1   101.165500   99.153376  100.073979          False         False   
2   101.012396   99.475928   99.898539          False         False   
3   100.919061   98.863476   99.738135          False          True   
4   100.609213   99.318854   99.886589          False         False   
5   100.609196   99.036234   99.895110          False         False   
6   100.540659   99.281802  100.137764          False         False   
7   101.106323   99.135952  100.082534           True         False   
8   100.920781   99.117303  100.232328          False         False   
9   100.995651   99.370602   99.983329          False         False   
10  100.514409   98.821291   99.771757          False          True   
11  102.678937  100.457407  101.627947          False         False   
12  104.082710  101.842351  103.156471          False         False   
13  10

In [6]:
import numpy as np
import pandas as pd

df = pd.read_csv("./Trading Data/GBPJPY_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df = df.loc['2026-05-01 01:00' : '2026-9-17 05:00']

timezone = "Europe/London"
df.index = pd.to_datetime(df.index, utc=True).tz_convert(timezone)

df_result = implement_market_structure_states_strict(df, n=2)
# print(df_result)

df['market_state'] = df_result["market_state"]

print(df)

# for index, row in df_result.iterrows():
#     print(row)

# Affichage des transitions d'états
df_result["state_changed"] = df_result["market_state"] != df_result[
    "market_state"
].shift(1)
print(df_result[df_result["state_changed"]][["Close", "market_state"]])

                              Open     High      Low    Close  Tickvol  \
Date                                                                     
2026-05-03 01:00:00+01:00  213.057  213.190  213.004  213.079     5145   
2026-05-04 01:00:00+01:00  213.081  213.613  211.756  212.720   216229   
2026-05-05 01:00:00+01:00  212.720  214.070  212.549  213.925   167413   
2026-05-06 01:00:00+01:00  213.923  214.224  210.747  212.723   296782   
2026-05-07 01:00:00+01:00  212.724  213.117  212.281  212.608   205257   
...                            ...      ...      ...      ...      ...   
2026-09-13 01:00:00+01:00  207.280  207.912  207.271  207.500    37103   
2026-09-14 01:00:00+01:00  207.503  208.767  207.455  208.468   387797   
2026-09-15 01:00:00+01:00  208.467  209.342  208.439  209.169   283811   
2026-09-16 01:00:00+01:00  209.168  209.467  208.339  208.807   287162   
2026-09-17 01:00:00+01:00  208.807  208.974  208.226  208.647   111232   

                          market_stat

In [4]:
import pandas as pd
import time

from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

global_range_to_consider = 0
global_tp_multiple = 0
global_stop_trade_hour = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    tp_multiple = 5.5
    stop_trade_hour = 18
    close_range_hour = 8

    # range_to_consider = 3
    # close_range_hour = 9
    # tp_multiple = 3
    # stop_trade_hour = 20

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.opening_range_high = None
        self.opening_range_low = None
        if global_range_to_consider != 0:
            print(global_range_to_consider)
            self.range_to_consider = global_range_to_consider

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_close_range_hour != 0:
            self.close_range_hour = global_close_range_hour

        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.opening_range_high = None
        self.opening_range_low = None

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high 
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_hour

            if not self.current_day_position_taked:
                if c_long and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price + per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
                if c_short and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True

df = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df = df.loc['2026-01-01 01:00' : '2026-9-17 05:00']

timezone = "Europe/London"
df.index = pd.to_datetime(df.index, utc=True).tz_convert(timezone)

df_result = implement_market_structure_states_strict(df, n=4)
# print(df_result)

df['market_state'] = df_result["market_state"]

indicator_atr_14 = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14)
df['ATR'] = indicator_atr_14._atr

bt = Backtest(df, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

Start                     2026-01-01 22:00...
End                       2026-09-17 06:00...
Duration                    258 days 07:00:00
Exposure Time [%]                    71.38983
Equity Final [$]                    11928.146
Equity Peak [$]                     13195.977
Return [%]                           19.28146
Buy & Hold Return [%]                -1.28489
Return (Ann.) [%]                    22.15764
Volatility (Ann.) [%]                33.20494
CAGR [%]                             28.31627
Sharpe Ratio                           0.6673
Sortino Ratio                         1.33687
Calmar Ratio                          1.10265
Alpha [%]                            19.92577
Beta                                  0.50145
Max. Drawdown [%]                   -20.09489
Avg. Drawdown [%]                    -2.47085
Max. Drawdown Duration      100 days 16:00:00
Avg. Drawdown Duration        8 days 10:00:00
# Trades                                  133
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1261', ...)

In [3]:
import pandas as pd
import time

from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

global_range_to_consider = 0
global_tp_multiple = 0
global_stop_trade_hour = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    tp_multiple = 5.5
    stop_trade_hour = 18
    close_range_hour = 8

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.opening_range_high = None
        self.opening_range_low = None
        if global_range_to_consider != 0:
            self.range_to_consider = global_range_to_consider

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_close_range_hour != 0:
            self.close_range_hour = global_close_range_hour

        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.opening_range_high = None
        self.opening_range_low = None

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['market_state'] == 'BULLISH'
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['market_state'] == 'BEARISH'
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_hour

            if not self.current_day_position_taked:
                if c_long and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price + per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
                if c_short and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True

In [6]:
df = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df = df.loc['2020-01-01 01:00' : '2026-9-17 05:00']

timezone = "Europe/London"
df.index = pd.to_datetime(df.index, utc=True).tz_convert(timezone)

df_result = implement_market_structure_states_strict(df, n=4)
# print(df_result)

df['market_state'] = df_result["market_state"]

indicator_atr_14 = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14)
df['ATR'] = indicator_atr_14._atr

bt = Backtest(df, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

Start                     2020-01-01 22:00...
End                       2026-09-17 06:00...
Duration                   2450 days 07:00:00
Exposure Time [%]                    57.05445
Equity Final [$]                    16969.378
Equity Peak [$]                     21638.773
Return [%]                           69.69378
Buy & Hold Return [%]                44.73288
Return (Ann.) [%]                     6.55803
Volatility (Ann.) [%]                18.70312
CAGR [%]                               8.2019
Sharpe Ratio                          0.35064
Sortino Ratio                         0.62936
Calmar Ratio                          0.22277
Alpha [%]                            60.63118
Beta                                  0.20259
Max. Drawdown [%]                   -29.43836
Avg. Drawdown [%]                    -2.04144
Max. Drawdown Duration      720 days 00:00:00
Avg. Drawdown Duration       20 days 08:00:00
# Trades                                  661
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1544', ...)

In [7]:
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

start_date = '2020-01-01 00:00'
end_date = '2026-9-17 00:00'

df_d1 = pd.read_csv("./Trading Data/GBPJPY_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

df_result = implement_market_structure_states_strict(df_d1, n=2)

# df['market_state'] = df_result["market_state"]

df_h1 = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

d1_tf_direction_list = []

for index, row in df_result.iterrows():
    df_by_day = df_h1.loc[f'{index.date()}' : f'{index.date()}']
    for index2, row2 in df_by_day.iterrows():
        d1_tf_direction_list.append(row['market_state'])

df_h1['market_state'] = d1_tf_direction_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr
timezone = "Europe/London"
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()


Start                     2020-01-01 22:00...
End                       2026-09-17 01:00...
Duration                   2450 days 02:00:00
Exposure Time [%]                    51.95708
Equity Final [$]                   33051.4975
Equity Peak [$]                    34924.7605
Return [%]                          230.51497
Buy & Hold Return [%]                45.09965
Return (Ann.) [%]                     15.4416
Volatility (Ann.) [%]                17.83355
CAGR [%]                             19.50862
Sharpe Ratio                          0.86587
Sortino Ratio                         1.77632
Calmar Ratio                          0.75816
Alpha [%]                           220.31218
Beta                                  0.22623
Max. Drawdown [%]                    -20.3673
Avg. Drawdown [%]                    -1.59435
Max. Drawdown Duration      530 days 13:00:00
Avg. Drawdown Duration       12 days 06:00:00
# Trades                                  495
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1828', ...)

In [6]:
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

start_date = '2020-01-01 00:00'
end_date = '2026-9-17 00:00'

df_d1 = pd.read_csv("./Trading Data/GBPJPY_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

df_h1 = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr
timezone = "Europe/London"

d1_window_list = [1,2,3,4]
range_to_consider_list = [3,4,5]
close_range_hour_list = [7,8,9]
stop_trade_hour_list = [18,19,20,21]
tp_multiple_list = [2,2.5,3,3.5,4,4.5,5,5.5,6]

with open(f"./results/result_GBPJPY_{time.time()}.txt", "a") as f:
    for window in d1_window_list:
        df_h1_test = df_h1.copy()
        df_result = implement_market_structure_states_strict(df_d1, n=window)

        d1_tf_direction_list = []

        for index, row in df_result.iterrows():
            df_by_day = df_h1_test.loc[f'{index.date()}' : f'{index.date()}']
            for index2, row2 in df_by_day.iterrows():
                d1_tf_direction_list.append(row['market_state'])
        
        df_h1_test['market_state'] = d1_tf_direction_list
        df_h1_test.index = pd.to_datetime(df_h1_test.index, utc=True).tz_convert(timezone)

        for range_to_consider in range_to_consider_list:
            for close_range_hour in close_range_hour_list:
                for stop_trade_hour in stop_trade_hour_list:
                    for tp_multiple in tp_multiple_list:
                        # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                        global_range_to_consider = range_to_consider
                        global_close_range_hour = close_range_hour
                        global_stop_trade_hour = stop_trade_hour
                        global_tp_multiple = tp_multiple

                        bt = Backtest(df_h1_test, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                        stats = bt.run()

                        f.write(f"d1 window = {window} & range to consider = {range_to_consider} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple}\n")
                        f.write(f"Return [%] = {stats['Return [%]']} | Return (Ann.) [%] = {stats['Return (Ann.) [%]']} | Max. Drawdown [%] = {stats['Max. Drawdown [%]']} | Win Rate [%] = {stats['Win Rate [%]']}\n")
                        f.write("----------------------------------------------\n")

                        print(f"d1 window = {window} & range to consider = {range_to_consider} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")



d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 2 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 2.5 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 3 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 3.5 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 4 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 4.5 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5.5 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 6 DONE
d1 window = 1 & range to cons

In [8]:
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

start_date = '2020-01-01 00:00'
end_date = '2026-9-17 00:00'

df_h4 = pd.read_csv("./Trading Data/GBPJPY_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4 = df_h4.loc[start_date : end_date]

df_result = implement_market_structure_states_strict(df_h4, n=5)

# df['market_state'] = df_result["market_state"]

df_h1 = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

d1_tf_direction_list = []

for index in range(len(df_result)):
    row = df_result.iloc[index]
    start_date = row.name
    end_date = start_date + timedelta(hours=3)
    df_by_day = df_h1.loc[f'{start_date}' : f'{end_date}']

    for index2, row2 in df_by_day.iterrows():
        d1_tf_direction_list.append(row['market_state'])


df_h1['market_state'] = d1_tf_direction_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr
timezone = "Europe/London"
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()


Start                     2020-01-01 22:00...
End                       2026-09-17 01:00...
Duration                   2450 days 02:00:00
Exposure Time [%]                    54.34429
Equity Final [$]                   19752.3495
Equity Peak [$]                     24599.015
Return [%]                           97.52349
Buy & Hold Return [%]                45.09965
Return (Ann.) [%]                     8.51957
Volatility (Ann.) [%]                17.85966
CAGR [%]                             10.68017
Sharpe Ratio                          0.47703
Sortino Ratio                         0.88931
Calmar Ratio                          0.33651
Alpha [%]                            90.56202
Beta                                  0.15436
Max. Drawdown [%]                   -25.31779
Avg. Drawdown [%]                    -2.32176
Max. Drawdown Duration      761 days 17:00:00
Avg. Drawdown Duration       23 days 11:00:00
# Trades                                  506
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p2112', ...)

In [ ]:
from datetime import datetime, timedelta
import time

import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

start_date = '2020-01-01 00:00'
end_date = '2026-9-17 00:00'

df_h4 = pd.read_csv("./Trading Data/GBPJPY_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4 = df_h4.loc[start_date : end_date]

df_h1 = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr
timezone = "Europe/London"

#6,7,8,9,10,11,12
h4_window_list = [8,9,10,11,12]
range_to_consider_list = [3,4,5]
close_range_hour_list = [7,8,9]
stop_trade_hour_list = [18,19,20,21]
tp_multiple_list = [2,2.5,3,3.5,4,4.5,5,5.5,6]

with open(f"./results/result_GBPJPY_{time.time()}.txt", "a") as f:
    for window in h4_window_list:
        df_h1_test = df_h1.copy()
        df_result = implement_market_structure_states_strict(df_h4, n=window)

        d1_tf_direction_list = []

        for index in range(len(df_result)):
            row = df_result.iloc[index]
            start_date = row.name
            end_date = start_date + timedelta(hours=3)
            df_by_day = df_h1_test.loc[f'{start_date}' : f'{end_date}']

            for index2, row2 in df_by_day.iterrows():
                d1_tf_direction_list.append(row['market_state'])
        
        df_h1_test['market_state'] = d1_tf_direction_list
        df_h1_test.index = pd.to_datetime(df_h1_test.index, utc=True).tz_convert(timezone)

        for range_to_consider in range_to_consider_list:
            for close_range_hour in close_range_hour_list:
                for stop_trade_hour in stop_trade_hour_list:
                    for tp_multiple in tp_multiple_list:
                        # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                        global_range_to_consider = range_to_consider
                        global_close_range_hour = close_range_hour
                        global_stop_trade_hour = stop_trade_hour
                        global_tp_multiple = tp_multiple

                        bt = Backtest(df_h1_test, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                        stats = bt.run()

                        f.write(f"h4 window = {window} & range to consider = {range_to_consider} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple}\n")
                        f.write(f"Return [%] = {stats['Return [%]']} | Return (Ann.) [%] = {stats['Return (Ann.) [%]']} | Max. Drawdown [%] = {stats['Max. Drawdown [%]']} | Win Rate [%] = {stats['Win Rate [%]']}\n")
                        f.write("----------------------------------------------\n")

                        print(f"h4 window = {window} & range to consider = {range_to_consider} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")



h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 2 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 2.5 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 3 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 3.5 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 4 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 4.5 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5.5 DONE
h4 window = 8 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 6 DONE
h4 window = 8 & range to cons

In [4]:
import pandas as pd
import numpy as np
import time

from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

start_date = '2020-01-01 00:00'
end_date = '2026-9-17 00:00'

df_d1 = pd.read_csv("./Trading Data/GBPJPY_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

df_h1 = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr
timezone = "Europe/London"

d1_window_list = [1,2]
range_to_consider_list = [3]
close_range_hour_list = [7]
stop_trade_hour_list = [18]
tp_multiple_list = [6]

with open(f"./results/result_GBPJPY_{time.time()}.txt", "a") as f:
    f.write(f"Window,RTC,CRH,STH,TpMultiple,Return,ReturnYearly,MDrawdown,WinRate,NumTrades\n")

    for window in d1_window_list:
        df_h1_test = df_h1.copy()
        df_result = implement_market_structure_states_strict(df_d1, n=window)

        d1_tf_direction_list = []

        for index, row in df_result.iterrows():
            df_by_day = df_h1_test.loc[f'{index.date()}' : f'{index.date()}']
            for index2, row2 in df_by_day.iterrows():
                d1_tf_direction_list.append(row['market_state'])
        
        df_h1_test['market_state'] = d1_tf_direction_list
        df_h1_test.index = pd.to_datetime(df_h1_test.index, utc=True).tz_convert(timezone)

        for range_to_consider in range_to_consider_list:
            for close_range_hour in close_range_hour_list:
                for stop_trade_hour in stop_trade_hour_list:
                    for tp_multiple in tp_multiple_list:
                        # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                        global_range_to_consider = range_to_consider
                        global_close_range_hour = close_range_hour
                        global_stop_trade_hour = stop_trade_hour
                        global_tp_multiple = tp_multiple

                        bt = Backtest(df_h1_test, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                        stats = bt.run()

                        f.write(f"{window},{range_to_consider},{close_range_hour},{stop_trade_hour},{tp_multiple},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']}, {stats['# Trades']}\n")

                        print(f"d1 window = {window} & range to consider = {range_to_consider} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")



d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 6 DONE
d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 6 DONE


In [5]:
df_result = pd.read_csv("./results/result_GBPJPY_1789884178.635549.txt",sep=',')

print(df_result)

   Window  RTC  CRH  STH  TpMultiple     Return  ReturnYearly  MDrawdown  \
0       1    3    7   18           6  189.28530     13.608791  -23.33679   
1       2    3    7   18           6  220.94442     15.034872  -17.61019   

     WinRate  NumTrades  
0  20.442930        587  
1  20.404412        544  
